In [ ]:

from pathlib import Path
import pandas as pd

DATA = Path("../data/ham10000")

# Read the table that describes every image (diagnosis, age, etc.)
meta = pd.read_csv(DATA / "HAM10000_metadata")   

paths = {p.stem: p for p in DATA.rglob("*.jpg")}
meta["path"] = meta["image_id"].map(paths)

print(meta.shape, "-", meta["path"].isna().sum(), "images missing")

meta["dx"].value_counts()

(10015, 9) - 0 images missing


dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

# Keep only the four classes that also exist in PAD-UFES-20
meta = meta[meta["dx"].isin(["nv", "bkl", "mel", "bcc"])].copy()
meta["label"] = meta["dx"].isin(["mel", "bcc"]).astype(int)

lesions = meta.drop_duplicates("lesion_id")[["lesion_id", "label"]]

# 70% train, then the other 30% split into 15% val and 15% test
train_l, temp_l = train_test_split(lesions, test_size=0.30, stratify=lesions["label"], random_state=SEED)
val_l, test_l = train_test_split(temp_l, test_size=0.50, stratify=temp_l["label"], random_state=SEED)

split_of = {**{i: "train" for i in train_l["lesion_id"]},
            **{i: "val" for i in val_l["lesion_id"]},
            **{i: "test" for i in test_l["lesion_id"]}}
meta["split"] = meta["lesion_id"].map(split_of)

assert meta.groupby("lesion_id")["label"].nunique().max() == 1
assert meta.groupby("lesion_id")["split"].nunique().max() == 1

# Show how many benign (0) and malignant (1) images are in each split
print(meta.groupby(["split", "label"]).size().unstack())

meta[["image_id", "lesion_id", "dx", "label", "split"]].to_csv("../results/ham10000_split.csv", index=False)

label     0     1
split            
test   1178   264
train  5449  1120
val    1177   243
